In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

from utils import to_gpu
import json
import os
import numpy as np

Two Decoder input sequence -> output squence (LSTM encoder)

Some notes: 
The encoder:  Reads tokens  Uses an LSTM  Produces a hidden vector
The decoder:  Takes a hidden vector  Generates a sentence word by word
The encoder:  Embeds words  Runs an LSTM  Takes the last hidden state  Normalizes it to unit length  Optionally adds noise
(mane ekta latent vector pabe sentence theke)
The decoder:  Takes that hidden vector  Concatenates it with word embeddings  Runs another LSTM  Predicts the next word at every step  Outputs vocabulary logits (mane probability of x given hidden is known??)
2 decoders because of step 6: pdecb​(xb​∣z,ha​,y)


In [ ]:
class Seq2Seq2Decoder(nn.Module):
    def __init__(self, emsize, nhidden, ntokens, nlayers, noise_r=0.2,
                 share_decoder_emb=False, hidden_init=False, dropout=0, gpu=False):
        super(Seq2Seq2Decoder, self).__init__()
        self.nhidden = nhidden
        self.emsize = emsize
        self.ntokens = ntokens
        self.nlayers = nlayers
        self.noise_r = noise_r
        self.hidden_init = hidden_init
        self.dropout = dropout
        
        # #gpu thingy
        # self.gpu = gpu
        # self.start_symbols = to_gpu(gpu, Variable(torch.ones(10, 1).long()))
        
        #embedding vocab
        self.embedding = nn.Embedding(ntokens, emsize)
        self.embedding_decoder1 = nn.Embedding(ntokens, emsize)
        self.embedding_decoder2 = nn.Embedding(ntokens, emsize)

        #rnn: first encoder, then decoder
        self.encoder = nn.LSTM(input_size=emsize,
                               hidden_size=nhidden,
                               num_layers=nlayers,
                               dropout=dropout,
                               batch_first=True)

        decoder_input_size = emsize+nhidden
        self.decoder1 = nn.LSTM(input_size=decoder_input_size,
                               hidden_size=nhidden,
                               num_layers=1,
                               dropout=dropout,
                               batch_first=True)
        self.decoder2 = nn.LSTM(input_size=decoder_input_size,
                               hidden_size=nhidden,
                               num_layers=1,
                               dropout=dropout,
                               batch_first=True)

        #linear trandsformation --- ??? 
        self.linear = nn.Linear(nhidden, ntokens)

        self.init_weights()
        if share_decoder_emb:
            self.embedding_decoder2.weight = self.embedding_decoder1.weight

    def init_weights(self):
        initrange = 0.1

        #vocab matrix
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.embedding_decoder1.weight.data.uniform_(-initrange, initrange)
        self.embedding_decoder2.weight.data.uniform_(-initrange, initrange)

        #encoder decoder init
        for p in self.encoder.parameters():
            p.data.uniform_(-initrange, initrange)
        for p in self.decoder1.parameters():
            p.data.uniform_(-initrange, initrange)
        for p in self.decoder2.parameters():
            p.data.uniform_(-initrange, initrange)

        #lin weight
        self.linear.weight.data.uniform_(-initrange, initrange)
        self.linear.bias.data.fill_(0)

    def init_hidden(self, bsz):
        zeros1 = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        zeros2 = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        return (to_gpu(self.gpu, zeros1), to_gpu(self.gpu, zeros2))

    def init_state(self, bsz):
        zeros = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        return to_gpu(self.gpu, zeros)

    def store_grad_norm(self, grad):
        norm = torch.norm(grad, 2, 1)
        self.grad_norm = norm.detach().data.mean()
        return grad

    def forward(self, whichdecoder, indices, lengths, noise=False, encode_only=False):
        batch_size, maxlen = indices.size()

        hidden = self.encode(indices, lengths, noise)

        if hidden.requires_grad:
            hidden.register_hook(self.store_grad_norm)

        if encode_only:
            return hidden
        decoded = self.decode(whichdecoder, hidden, batch_size, maxlen,
                              indices=indices, lengths=lengths)

        return decoded

    def encode(self, indices, lengths, noise):
        embeddings = self.embedding(indices)
        packed_embeddings = pack_padded_sequence(input=embeddings,
                                                 lengths=lengths,
                                                 batch_first=True)

        #encoder call
        packed_output, state = self.encoder(packed_embeddings)
        hidden, cell = state

        #batch_size*nhidden, normalize to unit sphere (hidden/||hidden||2)
        hidden = hidden[-1]  #last layer hidden state
        norms = torch.norm(hidden, 2, 1)

        if noise and self.noise_r > 0:
            gauss_noise = torch.normal(mean=torch.zeros(hidden.size()),
                                       std=self.noise_r)
            hidden = hidden + to_gpu(self.gpu, Variable(gauss_noise))

        return hidden

    def decode(self, whichdecoder, hidden, batch_size, maxlen, indices=None, lengths=None):
        #batch*hidden
        all_hidden = hidden.unsqueeze(1).repeat(1, maxlen, 1)

        if self.hidden_init:
            #initialize decoder hidden state to encoder output
            state = (hidden.unsqueeze(0), self.init_state(batch_size))
        else:
            state = self.init_hidden(batch_size)

        if whichdecoder == 1:
            embeddings = self.embedding_decoder1(indices)
        else:
            embeddings = self.embedding_decoder2(indices)

        augmented_embeddings = torch.cat([embeddings, all_hidden], 2)   #this is [z; ha] concatenation
        packed_embeddings = pack_padded_sequence(input=augmented_embeddings,
                                                 lengths=lengths,
                                                 batch_first=True)

        if whichdecoder == 1:
            packed_output, state = self.decoder1(packed_embeddings, state)
        else:
            packed_output, state = self.decoder2(packed_embeddings, state)
        output, lengths = pad_packed_sequence(packed_output, batch_first=True)

        #reshape to batch_size*maxlen*nhidden before linear over vocab
        decoded = self.linear(output.contiguous().view(-1, self.nhidden))
        decoded = decoded.view(batch_size, maxlen, self.ntokens)

        return decoded

    def generate(self, whichdecoder, hidden, maxlen, sample=False, temp=1.0):
        """Generate through decoder; no backprop"""

        batch_size = hidden.size(0)

        if self.hidden_init:
            #initialize decoder hidden state to encoder output
            state = (hidden.unsqueeze(0), self.init_state(batch_size))
        else:
            state = self.init_hidden(batch_size)

        # <sos>
        self.start_symbols.resize_(batch_size, 1) 
        self.start_symbols.fill_(1) 
        self.start_symbols = to_gpu(self.gpu, self.start_symbols)

        if whichdecoder == 1:
            embedding = self.embedding_decoder1(self.start_symbols)
        else:
            embedding = self.embedding_decoder2(self.start_symbols)

        inputs = torch.cat([embedding, hidden.unsqueeze(1)], 2)

        #unroll
        all_indices = []
        for i in range(maxlen):
            if whichdecoder == 1:
                output, state = self.decoder1(inputs, state)
            else:
                output, state = self.decoder2(inputs, state)
            overvocab = self.linear(output.squeeze(1))
            
            if not sample:
                vals, indices = torch.max(overvocab, 1)
                indices = indices.unsqueeze(1)
            else:
                assert 1 == 0
                # sampling
                probs = F.softmax(overvocab/temp)
                indices = torch.multinomial(probs, 1)

            all_indices.append(indices)

            if whichdecoder == 1:      #decoder choosing
                embedding = self.embedding_decoder1(indices)
            else:
                embedding = self.embedding_decoder2(indices)
            inputs = torch.cat([embedding, hidden.unsqueeze(1)], 2)

        max_indices = torch.cat(all_indices, 1)

        return max_indices


Two Decoder input sequence -> output squence (CNN encoder)

Encoder: Words → embeddings (self.embedding)  Transpose → needed for CNN input (channels first)  Pass through CNN layers → extract features from local word patterns  Pooling → get one fixed-size vector per sentence  Linear layer → map to latent space (emsize)  Normalize → unit vector  Optionally add Gaussian noise (mane oi same input sentence to latent vector)
Decoder (LSTM): Two decoders exist, just like before (decoder1 and decoder2)  Input to the decoder is latent vector + word embeddings at each time step  LSTM predicts the next word sequentially  Output linear layer → vocabulary logits (basically same)

In [ ]:
class Seq2Seq2CNNDecoder(nn.Module):
    def __init__(self, emsize, nhidden, ntokens, nlayers, conv_windows="5-5-3", conv_strides="2-2-2",
                 conv_layer="500-700-1000", activation=nn.LeakyReLU(0.2, inplace=True), noise_r=0.2,
                 share_decoder_emb=False, hidden_init=False, dropout=0, gpu=False, pooling_enc="avg"):
        super(Seq2Seq2CNNDecoder, self).__init__()
        self.nhidden = nhidden
        self.emsize = emsize
        self.ntokens = ntokens
        self.nlayers = nlayers
        self.noise_r = noise_r
        self.hidden_init = hidden_init
        self.dropout = dropout
        self.gpu = gpu
        # for CNN encoder
        self.arch_conv_filters = conv_layer
        self.arch_conv_strides = conv_strides
        self.arch_conv_windows = conv_windows

        ##gpu thingy
        #self.start_symbols = to_gpu(gpu, Variable(torch.ones(10, 1).long()))

        # Vocabulary embedding
        self.embedding = nn.Embedding(ntokens, emsize)
        self.embedding_decoder1 = nn.Embedding(ntokens, emsize)
        self.embedding_decoder2 = nn.Embedding(ntokens, emsize)

        # for CNN encoder
        conv_layer_sizes = [emsize] + [int(x) for x in conv_layer.split('-')]
        conv_strides_sizes = [int(x) for x in conv_strides.split('-')]
        conv_windows_sizes = [int(x) for x in conv_windows.split('-')]
        self.encoder = nn.Sequential()

        for i in range(len(conv_layer_sizes) - 1):
            layer = nn.Conv1d(conv_layer_sizes[i], conv_layer_sizes[i + 1], \
                              conv_windows_sizes[i], stride=conv_strides_sizes[i])
            self.encoder.add_module("layer-" + str(i + 1), layer)

            bn = nn.BatchNorm1d(conv_layer_sizes[i + 1])
            self.encoder.add_module("bn-" + str(i + 1), bn)

            self.encoder.add_module("activation-" + str(i + 1), activation)

        if pooling_enc == "max":
            self.pooling_enc = nn.AdaptiveMaxPool1d(1)  #or gbmaxpool?
        else:
            self.pooling_enc = nn.AdaptiveAvgPool1d(1)
        self.linear_enc = nn.Linear(1000, emsize)

        decoder_input_size = emsize+nhidden
        self.decoder1 = nn.LSTM(input_size=decoder_input_size,
                               hidden_size=nhidden,
                               num_layers=1,
                               dropout=dropout,
                               batch_first=True)
        self.decoder2 = nn.LSTM(input_size=decoder_input_size,
                               hidden_size=nhidden,
                               num_layers=1,
                               dropout=dropout,
                               batch_first=True)

        # Initialize Linear Transformation
        self.linear = nn.Linear(nhidden, ntokens)

        self.init_weights()

        if share_decoder_emb:
            self.embedding_decoder2.weight = self.embedding_decoder1.weight

    def init_weights(self):
        initrange = 0.1

        # Initialize Vocabulary Matrix Weight
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.embedding_decoder1.weight.data.uniform_(-initrange, initrange)
        self.embedding_decoder2.weight.data.uniform_(-initrange, initrange)

        # Initialize Encoder and Decoder Weights
        for p in self.encoder.parameters():
            p.data.uniform_(-initrange, initrange)
        for p in self.decoder1.parameters():
            p.data.uniform_(-initrange, initrange)
        for p in self.decoder2.parameters():
            p.data.uniform_(-initrange, initrange)

        # Initialize Linear Weight
        self.linear.weight.data.uniform_(-initrange, initrange)
        self.linear.bias.data.fill_(0)

        self.linear_enc.weight.data.uniform_(-initrange, initrange)
        self.linear_enc.bias.data.fill_(0)

    def init_hidden(self, bsz):
        zeros1 = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        zeros2 = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        return (to_gpu(self.gpu, zeros1), to_gpu(self.gpu, zeros2))

    def init_state(self, bsz):
        zeros = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        return to_gpu(self.gpu, zeros)

    def store_grad_norm(self, grad):
        norm = torch.norm(grad, 2, 1)
        self.grad_norm = norm.detach().data.mean()
        return grad

    def forward(self, whichdecoder, indices, lengths, noise=False, encode_only=False):
        batch_size, maxlen = indices.size()

        hidden = self.encode(indices, lengths, noise)

        if hidden.requires_grad:
            hidden.register_hook(self.store_grad_norm)

        if encode_only:
            return hidden

        decoded = self.decode(whichdecoder, hidden, batch_size, maxlen,
                              indices=indices, lengths=lengths)

        return decoded

    def encode(self, indices, lengths, noise):
        embeddings = self.embedding(indices)
        embeddings = embeddings.transpose(1,2)
        c_pre_lin = self.encoder(embeddings)
        c_pre_lin = self.pooling_enc(c_pre_lin)
        c_pre_lin = c_pre_lin.squeeze(2)
        hidden = self.linear_enc(c_pre_lin)
        # normalize to unit ball (l2 norm of 1) - p=2, dim=1
        norms = torch.norm(hidden, 2, 1)
        if norms.ndimension()==1:
            norms=norms.unsqueeze(1)
        hidden = torch.div(hidden, norms.expand_as(hidden))

        if noise and self.noise_r > 0:
            gauss_noise = torch.normal(mean=torch.zeros(hidden.size()),
                                       std=self.noise_r)
            if self.gpu:
                gauss_noise = gauss_noise.cuda()
                
            hidden = hidden + to_gpu(self.gpu, Variable(gauss_noise))

        return hidden

    def decode(self, whichdecoder, hidden, batch_size, maxlen, indices=None, lengths=None):
        # batch x hidden
        all_hidden = hidden.unsqueeze(1).repeat(1, maxlen, 1)

        if self.hidden_init:
            # initialize decoder hidden state to encoder output
            state = (hidden.unsqueeze(0), self.init_state(batch_size))
        else:
            state = self.init_hidden(batch_size)

        if whichdecoder == 1:
            embeddings = self.embedding_decoder1(indices)
        else:
            embeddings = self.embedding_decoder2(indices)

        augmented_embeddings = torch.cat([embeddings, all_hidden], 2)
        packed_embeddings = pack_padded_sequence(input=augmented_embeddings,
                                                 lengths=lengths,
                                                 batch_first=True)

        if whichdecoder == 1:
            packed_output, state = self.decoder1(packed_embeddings, state)
        else:
            packed_output, state = self.decoder2(packed_embeddings, state)
        output, lengths = pad_packed_sequence(packed_output, batch_first=True)

        # reshape to batch_size*maxlen x nhidden before linear over vocab
        decoded = self.linear(output.contiguous().view(-1, self.nhidden))
        decoded = decoded.view(batch_size, maxlen, self.ntokens)

        return decoded

    def generate(self, whichdecoder, hidden, maxlen, sample=False, temp=1.0):
        """Generate through decoder; no backprop"""

        batch_size = hidden.size(0)

        if self.hidden_init:
            # initialize decoder hidden state to encoder output
            state = (hidden.unsqueeze(0), self.init_state(batch_size))
        else:
            state = self.init_hidden(batch_size)

        # <sos>
        self.start_symbols.resize_(batch_size, 1) 
        self.start_symbols.fill_(1) 
        self.start_symbols = to_gpu(self.gpu, self.start_symbols)

        if whichdecoder == 1:
            embedding = self.embedding_decoder1(self.start_symbols)
        else:
            embedding = self.embedding_decoder2(self.start_symbols)

        inputs = torch.cat([embedding, hidden.unsqueeze(1)], 2)

        # unroll
        all_indices = []
        for i in range(maxlen):
            if whichdecoder == 1:
                output, state = self.decoder1(inputs, state)
            else:
                output, state = self.decoder2(inputs, state)
            overvocab = self.linear(output.squeeze(1))
            
            if not sample:
                vals, indices = torch.max(overvocab, 1)
                indices = indices.unsqueeze(1)
            else:
                assert 1 == 0
                # sampling
                probs = F.softmax(overvocab/temp)
                indices = torch.multinomial(probs, 1)

            all_indices.append(indices)

            if whichdecoder == 1:
                embedding = self.embedding_decoder1(indices)
            else:
                embedding = self.embedding_decoder2(indices)
            inputs = torch.cat([embedding, hidden.unsqueeze(1)], 2)

        max_indices = torch.cat(all_indices, 1)

        return max_indices



Classifier

This function implements the latent classifier fclass, which predicts the target label from the encoded hidden representations z and ha. It allows the model to measure and control how much label information is encoded in the latent space during training.

In [ ]:

class MLP_Classify(nn.Module):
    def __init__(self, ninput, noutput, layers,
                 activation=nn.ReLU(), gpu=False):  #check the gpu thingy
        super(MLP_Classify, self).__init__()
        self.ninput = ninput
        self.noutput = noutput

        layer_sizes = [ninput] + [int(x) for x in layers.split('-')]
        self.layers = []

        for i in range(len(layer_sizes)-1):
            layer = nn.Linear(layer_sizes[i], layer_sizes[i+1])
            self.layers.append(layer)
            self.add_module("layer"+str(i+1), layer)

            #first lauer has no batch normalisation
            if i != 0:
                bn = nn.BatchNorm1d(layer_sizes[i+1])
                self.layers.append(bn)
                self.add_module("bn"+str(i+1), bn)

            self.layers.append(activation)
            self.add_module("activation"+str(i+1), activation)

        layer = nn.Linear(layer_sizes[-1], noutput)
        self.layers.append(layer)
        self.add_module("layer"+str(len(self.layers)), layer)

        self.init_weights()

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)        #sigmoid??
        return x

    def init_weights(self):
        init_std = 0.02
        for layer in self.layers:
            try:
                layer.weight.data.normal_(0, init_std)
                layer.bias.data.fill_(0)
            except:
                pass

Discriminator

Discriminator  Input: latent vector  z (from enc_b) or generated vector  z (from gen)  Output: scalar (after torch.mean(x) in forward)  (checking real or fake vector) 
The discriminator outputs a vector because of the final linear layer. That scalar is what the adversarial loss uses

In [ ]:
class MLP_D(nn.Module):
    def __init__(self, ninput, noutput, layers,
                 activation=nn.LeakyReLU(0.2), gpu=False):  #gpu check
        super(MLP_D, self).__init__()
        self.ninput = ninput
        self.noutput = noutput

        layer_sizes = [ninput] + [int(x) for x in layers.split('-')]
        self.layers = []

        for i in range(len(layer_sizes)-1):
            layer = nn.Linear(layer_sizes[i], layer_sizes[i+1])
            self.layers.append(layer)
            self.add_module("layer"+str(i+1), layer)

            #no batch normalization after first layer
            if i != 0:
                bn = nn.BatchNorm1d(layer_sizes[i+1], eps=1e-05, momentum=0.1)
                self.layers.append(bn)
                self.add_module("bn"+str(i+1), bn)

            self.layers.append(activation)
            self.add_module("activation"+str(i+1), activation)

        layer = nn.Linear(layer_sizes[-1], noutput)
        self.layers.append(layer)
        self.add_module("layer"+str(len(self.layers)), layer)

        self.init_weights()

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
        x = torch.mean(x)
        return x

    def init_weights(self):
        init_std = 0.02
        for layer in self.layers:
            try:
                layer.weight.data.normal_(0, init_std)
                layer.bias.data.fill_(0)
            except:
                pass


Generator

Input: random noise 
s∼N(0,I) Output: vector z that looks like real latent vectors

Purpose: generate latent vectors from noise to match the encoder’s latent distribution
MLP_D → judges real vs fake latent codes

In [ ]:
class MLP_G(nn.Module):
    def __init__(self, ninput, noutput, layers,
                 activation=nn.ReLU(), gpu=False):
        super(MLP_G, self).__init__()
        self.ninput = ninput
        self.noutput = noutput

        layer_sizes = [ninput] + [int(x) for x in layers.split('-')]
        self.layers = []

        for i in range(len(layer_sizes)-1):
            layer = nn.Linear(layer_sizes[i], layer_sizes[i+1])
            self.layers.append(layer)
            self.add_module("layer"+str(i+1), layer)

            bn = nn.BatchNorm1d(layer_sizes[i+1], eps=1e-05, momentum=0.1)
            self.layers.append(bn)
            self.add_module("bn"+str(i+1), bn)

            self.layers.append(activation)
            self.add_module("activation"+str(i+1), activation)

        layer = nn.Linear(layer_sizes[-1], noutput)
        self.layers.append(layer)
        self.add_module("layer"+str(len(self.layers)), layer)

        self.init_weights()

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
        return x

    def init_weights(self):
        init_std = 0.02
        for layer in self.layers:
            try:
                layer.weight.data.normal_(0, init_std)
                layer.bias.data.fill_(0)
            except:
                pass



Sequence to sequence (autoencoder+generate) - inference

This is the basic sequence-to-sequence autoencoder.  It has an encoder (LSTM) and a decoder (LSTM).  Encodes x_b or x_a → latent vector hidden  Decodes latent vector → reconstructed sequence 

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, emsize, nhidden, ntokens, nlayers, noise_r=0.2,
                 hidden_init=False, dropout=0, gpu=False):   #gpu check
        super(Seq2Seq, self).__init__()
        self.nhidden = nhidden
        self.emsize = emsize
        self.ntokens = ntokens
        self.nlayers = nlayers
        self.noise_r = noise_r
        self.hidden_init = hidden_init
        self.dropout = dropout
        self.gpu = gpu

        self.start_symbols = to_gpu(gpu, Variable(torch.ones(10, 1).long()))

        #vocabulary embedding
        self.embedding = nn.Embedding(ntokens, emsize)
        self.embedding_decoder = nn.Embedding(ntokens, emsize)

        #encoder decoder lstm
        self.encoder = nn.LSTM(input_size=emsize,
                               hidden_size=nhidden,
                               num_layers=nlayers,
                               dropout=dropout,
                               batch_first=True)

        decoder_input_size = emsize+nhidden
        self.decoder = nn.LSTM(input_size=decoder_input_size,
                               hidden_size=nhidden,
                               num_layers=1,
                               dropout=dropout,
                               batch_first=True)

        #linear transformation --?
        self.linear = nn.Linear(nhidden, ntokens)

        self.init_weights()

    def init_weights(self):
        initrange = 0.1

        #vocab matrix wt
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.embedding_decoder.weight.data.uniform_(-initrange, initrange)

        #encoder decoder weights
        for p in self.encoder.parameters():
            p.data.uniform_(-initrange, initrange)
        for p in self.decoder.parameters():
            p.data.uniform_(-initrange, initrange)

        #lin wts
        self.linear.weight.data.uniform_(-initrange, initrange)
        self.linear.bias.data.fill_(0)

    def init_hidden(self, bsz):
        zeros1 = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        zeros2 = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        return (to_gpu(self.gpu, zeros1), to_gpu(self.gpu, zeros2))

    def init_state(self, bsz):
        zeros = Variable(torch.zeros(self.nlayers, bsz, self.nhidden))
        return to_gpu(self.gpu, zeros)

    def store_grad_norm(self, grad):
        norm = torch.norm(grad, 2, 1)
        self.grad_norm = norm.detach().data.mean()
        return grad

    def forward(self, indices, lengths, noise, encode_only=False):
        batch_size, maxlen = indices.size()

        hidden = self.encode(indices, lengths, noise)

        if encode_only:
            return hidden

        if hidden.requires_grad:
            hidden.register_hook(self.store_grad_norm)

        decoded = self.decode(hidden, batch_size, maxlen,
                              indices=indices, lengths=lengths)

        return decoded

    def encode(self, indices, lengths, noise):
        embeddings = self.embedding(indices)
        packed_embeddings = pack_padded_sequence(input=embeddings,
                                                 lengths=lengths,
                                                 batch_first=True)

        packed_output, state = self.encoder(packed_embeddings)

        hidden, cell = state
        #batch_size*nhidden
        hidden = hidden[-1]  # get hidden state of last layer of encoder

        # normalize to unit sphere - p=2, dim=1
        norms = torch.norm(hidden, 2, 1)
        
        if noise and self.noise_r > 0:
            gauss_noise = torch.normal(mean=torch.zeros(hidden.size()),
                                       std=self.noise_r)
            hidden = hidden + to_gpu(self.gpu, Variable(gauss_noise))

        return hidden

    def decode(self, hidden, batch_size, maxlen, indices=None, lengths=None):
        # batch*hidden
        all_hidden = hidden.unsqueeze(1).repeat(1, maxlen, 1)

        if self.hidden_init:
            # initialize decoder hidden state to encoder output
            state = (hidden.unsqueeze(0), self.init_state(batch_size))
        else:
            state = self.init_hidden(batch_size)

        embeddings = self.embedding_decoder(indices)
        augmented_embeddings = torch.cat([embeddings, all_hidden], 2)
        packed_embeddings = pack_padded_sequence(input=augmented_embeddings,
                                                 lengths=lengths,
                                                 batch_first=True)

        packed_output, state = self.decoder(packed_embeddings, state)
        output, lengths = pad_packed_sequence(packed_output, batch_first=True)

        # reshape to batch_size*maxlen*nhidden before linear over vocab
        decoded = self.linear(output.contiguous().view(-1, self.nhidden))
        decoded = decoded.view(batch_size, maxlen, self.ntokens)

        return decoded

    def generate(self, hidden, maxlen, sample=False, temp=1.0):
        """Generate through decoder; no backprop"""

        batch_size = hidden.size(0)

        if self.hidden_init:
            # initialize decoder hidden state to encoder output
            state = (hidden.unsqueeze(0), self.init_state(batch_size))
        else:
            state = self.init_hidden(batch_size)

        # <sos>
        self.start_symbols.resize_(batch_size, 1) 
        self.start_symbols.fill_(1) 

        embedding = self.embedding_decoder(self.start_symbols)
        inputs = torch.cat([embedding, hidden.unsqueeze(1)], 2)

        # unroll
        all_indices = []
        for i in range(maxlen):
            output, state = self.decoder(inputs, state)
            overvocab = self.linear(output.squeeze(1))

            if not sample:
                vals, indices = torch.max(overvocab, 1)
            else:
                # sampling
                probs = F.softmax(overvocab/temp)
                indices = torch.multinomial(probs, 1)

            all_indices.append(indices)

            embedding = self.embedding_decoder(indices)
            inputs = torch.cat([embedding, hidden.unsqueeze(1)], 2)

        max_indices = torch.cat(all_indices, 1)

        return max_indices

#twodecoders - decoder choice; loads model arguments (args.json) and vocabulary (vocab.json)
def load_models(load_path, epoch, twodecoders=False):
    model_args = json.load(open("{}/args.json".format(load_path), "r"))
    word2idx = json.load(open("{}/vocab.json".format(load_path), "r"))
    idx2word = {v: k for k, v in word2idx.items()}

    if not twodecoders:
        autoencoder = Seq2Seq(emsize=model_args['emsize'],
                              nhidden=model_args['nhidden'],
                              ntokens=model_args['ntokens'],
                              nlayers=model_args['nlayers'],
                              hidden_init=model_args['hidden_init'])
    else:
        autoencoder = Seq2Seq2Decoder(emsize=model_args['emsize'],
                              nhidden=model_args['nhidden'],
                              ntokens=model_args['ntokens'],
                              nlayers=model_args['nlayers'],
                              hidden_init=model_args['hidden_init'])

    gan_gen = MLP_G(ninput=model_args['z_size'],
                    noutput=model_args['nhidden'],
                    layers=model_args['arch_g'])
    gan_disc = MLP_D(ninput=model_args['nhidden'],
                     noutput=1,
                     layers=model_args['arch_d'])

    print('Loading models from'+load_path)
    ae_path = os.path.join(load_path, "autoencoder_model_{}.pt".format(epoch))
    gen_path = os.path.join(load_path, "gan_gen_model_{}.pt".format(epoch))
    disc_path = os.path.join(load_path, "gan_disc_model_{}.pt".format(epoch))

    autoencoder.load_state_dict(torch.load(ae_path))
    gan_gen.load_state_dict(torch.load(gen_path))
    gan_disc.load_state_dict(torch.load(disc_path))
    return model_args, idx2word, autoencoder, gan_gen, gan_disc

#convert input z into a PyTorch var (if needed), set gan_gen and autoencoder to evaluation mode  Use the generator (MLP_G) to produce a latent vector fake_hidden, feed fake_hidden to the autoencoder’s decoder, process and return sequence
def generate(autoencoder, gan_gen, z, vocab, sample, maxlen):
    """
    Assume noise is batch_size x z_size
    """
    if type(z) == Variable:
        noise = z
    elif type(z) == torch.FloatTensor or type(z) == torch.cuda.FloatTensor:
        noise = Variable(z, volatile=True)
    elif type(z) == np.ndarray:
        noise = Variable(torch.from_numpy(z).float(), volatile=True)
    else:
        raise ValueError("Unsupported input type (noise): {}".format(type(z)))

    gan_gen.eval()
    autoencoder.eval()

    # generate from random noise
    fake_hidden = gan_gen(noise)
    max_indices = autoencoder.generate(hidden=fake_hidden,
                                       maxlen=maxlen,
                                       sample=sample)

    max_indices = max_indices.data.cpu().numpy()
    sentences = []
    for idx in max_indices:
        #generated sentence
        words = [vocab[x] for x in idx]
        #truncate sentences to first occurrence of <eos>
        truncated_sent = []
        for w in words:
            if w != '<eos>':
                truncated_sent.append(w)
            else:
                break
        sent = " ".join(truncated_sent)
        sentences.append(sent)

    return sentences


End Notes: A GAN is a system with two networks: a generator, which makes fake data from random noise, and a discriminator, which tries to tell real from fake. They compete: the generator improves to fool the discriminator, and the discriminator improves to catch fakes, so the generator eventually produces realistic outputs.

Here’s a clean, simple step-by-step description of what the code should do:

Step 0: The inputs are paired sequences, x_a (premise) and x_b (hypothesis), with labels y. The goal is to generate sequences like x_b conditioned on x_a and learn a meaningful latent space for x_b.

Step 1: The encoders convert input sequences into latent vectors. The decoder reconstructs x_b from the latent vector and the encoded x_a. Reconstruction loss measures how close the output is to the original x_b. Noise can be added to the latent space for robustness.

Step 2: A latent classifier optionally predicts the label y from the latent vector and x_a’s hidden state, encouraging the latent space to capture label-relevant information.

Step 3: A generator maps random noise to latent vectors. A discriminator tries to distinguish real latent vectors from generated ones. The generator learns to fool the discriminator, creating a structured latent space. This is the GAN component.

Step 4: For generation, random noise is fed into the generator to produce a latent vector. This vector, combined with the encoded x_a, is decoded to produce a sequence. Output indices are mapped to words and truncated at the end-of-sequence token <eos>.

Step 5: The code uses embeddings to convert tokens to vectors, batch normalization and activation functions to stabilize training, and optional CNN or multiple decoders for flexible encoding and decoding. Noise is used to regularize the latent space.